In [2]:
import datetime
import json
import re
import requests

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from bs4 import BeautifulSoup
import pandas as pd
import pytz
import talib

## fetch Alpaca's pricing data

In [3]:
with open('./Alpaca API Key.txt') as file:
    API_KEY = file.readline()
with open('./Alpaca API Secret.txt') as file:
    API_SECRET = file.readline()
    
client = StockHistoricalDataClient(API_KEY, API_SECRET)

In [4]:
def fetch_pricing_data(client, symbol, start, end):
    request_params = StockBarsRequest(symbol_or_symbols=[f'{symbol}'], timeframe=TimeFrame.Minute, start=start, end=end) # note that Alpaca returns all data from IEX in UTC timestamps
    bars = client.get_stock_bars(request_params)
    df = bars.df.reset_index()
    df['timestamp'] = df['timestamp'].apply(lambda x: x.astimezone(pytz.timezone('EST'))) # Eastern Standard Time (EST) is 5 hours behind Coordinated Universal Time (UTC)
    df.drop(columns=['symbol', 'trade_count', 'vwap'], inplace=True)
    return df

In [5]:
%%time
# symbol = 'TSLA'
symbol = 'ASML'
df_symbol = fetch_pricing_data(client, symbol, datetime.datetime(2016, 11, 25, 14, 30, tzinfo=datetime.timezone.utc), datetime.datetime(2023, 11, 25, 21, 00, tzinfo=datetime.timezone.utc))
df_symbol.head()

CPU times: user 11.5 s, sys: 10.9 s, total: 22.4 s
Wall time: 2min 34s


,timestamp,open,high,low,close,volume
0,2016-11-25 09:30:00-05:00,103.970,103.99,103.920,103.93,30594.0
1,2016-11-25 09:31:00-05:00,103.915,103.99,103.915,103.99,5533.0
2,2016-11-25 09:32:00-05:00,103.990,104.08,103.970,104.08,4530.0
3,2016-11-25 09:33:00-05:00,104.090,104.09,104.090,104.09,133.0
4,2016-11-25 09:34:00-05:00,104.020,104.02,103.940,103.94,2444.0


### add technical indicators

In [8]:
# https://ta-lib.github.io/ta-lib-python/funcs.html
df_symbol['EMA3'] = talib.EMA(df_symbol['close'], timeperiod=3)
df_symbol['EMA5'] = talib.EMA(df_symbol['close'], timeperiod=5)
df_symbol['EMA10'] = talib.EMA(df_symbol['close'], timeperiod=10)
df_symbol['EMA15'] = talib.EMA(df_symbol['close'], timeperiod=15)
df_symbol['RSI'] = talib.RSI(df_symbol['close'], timeperiod=14)
df_symbol['MACD'], df_symbol['MACD_signal'], df_symbol['MACD_hist'] = talib.MACD(df_symbol['close'], fastperiod=12, slowperiod=26, signalperiod=9)
df_symbol['MFI'] = talib.MFI(df_symbol['high'], df_symbol['low'], df_symbol['close'], df_symbol['volume'], timeperiod=14)
df_symbol['DX'] = talib.DX(df_symbol['high'], df_symbol['low'], df_symbol['close'], timeperiod=14)


In [9]:
df_symbol.head(40)

,timestamp,open,high,low,close,volume,EMA3,EMA5,EMA10,EMA15,RSI,MACD,MACD_signal,MACD_hist,MFI,DX
0,2016-11-25 09:30:00-05:00,103.9700,103.9900,103.9200,103.9300,30594.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-11-25 09:31:00-05:00,103.9150,103.9900,103.9150,103.9900,5533.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-11-25 09:32:00-05:00,103.9900,104.0800,103.9700,104.0800,4530.0,104.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-11-25 09:33:00-05:00,104.0900,104.0900,104.0900,104.0900,133.0,104.045000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-11-25 09:34:00-05:00,104.0200,104.0200,103.9400,103.9400,2444.0,103.992500,104.006000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2016-11-25 09:35:00-05:00,103.9800,103.9800,103.8900,103.8900,2900.0,103.941250,103.967333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2016-11-25 09:36:00-05:00,103.9700,104.0400,103.9400,104.0000,3680.0,103.970625,103.978222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2016-11-25 09:37:00-05:00,104.0300,104.0600,104.0300,104.0500,2676.0,104.010312,104.002148,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2016-11-25 09:38:00-05:00,104.0600,104.1000,104.0600,104.1000,1633.0,104.055156,104.034765,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2016-11-25 09:39:00-05:00,104.1000,104.1000,104.0300,104.0400,1350.0,104.047578,104.036510,104.011000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [77]:
df_symbol.iloc[33:].to_csv(f'./{symbol}.csv', index=False)

In [10]:
df_market = fetch_pricing_data(client, 'QQQ', df_symbol['timestamp'].iloc[0], df_symbol['timestamp'].iloc[-1])
df_market.head()

,timestamp,open,high,low,close,volume
0,2016-11-25 09:30:00-05:00,118.5700,118.650,118.56,118.60,287813.0
1,2016-11-25 09:31:00-05:00,118.6000,118.610,118.47,118.48,105250.0
2,2016-11-25 09:32:00-05:00,118.5200,118.570,118.49,118.57,71296.0
3,2016-11-25 09:33:00-05:00,118.5781,118.593,118.50,118.56,93718.0
4,2016-11-25 09:34:00-05:00,118.5600,118.560,118.43,118.46,44941.0


In [11]:
df_market.iloc[33:].to_csv(f'./QQQ.csv', index=False)

## fetch Alpaca's news data

In [81]:
def clean_text(text):
    # remove HTML tags using BeautifulSoup
    text_no_html = BeautifulSoup(text, 'html.parser').get_text(separator=' ')
    # remove URLs
    text_no_urls = re.sub(r'http\S+', '', text_no_html)
    # additional cleanup if needed (like removing Markdown or special characters)
    text_clean = re.sub(r'\[.*?\]\(.*?\)', '', text_no_urls)
    return text_clean

In [90]:
# https://docs.alpaca.markets/reference/news-1
url = "https://data.alpaca.markets/v1beta1/news?start=2016-11-25T14%3A30%3A00Z&end=2023-11-25T21%3A00%3A00Z&sort=asc&symbols=ASML&limit=50&include_content=true"
headers = {
    "accept": "application/json",
    "APCA-API-KEY-ID": API_KEY,
    "APCA-API-SECRET-KEY": API_SECRET
}

df_news = pd.DataFrame(columns=['text', 'timestamp'])
num_page = 1
response = json.loads(requests.get(url, headers=headers).text)
for news in response['news']:
    df_news.loc[len(df_news)] = [clean_text('\n'.join([news['headline'], news['summary'], news['content']])), news['created_at']]
print(f"page num: {num_page}; news num: {len(response['news'])}")
        
while response['next_page_token'] is not None:
    response = json.loads(requests.get(url + f"&page_token={response['next_page_token']}", headers=headers).text)
    num_page += 1
    for news in response['news']:
        df_news.loc[len(df_news)] = [clean_text(news['headline'] + '.' + news['content']), news['created_at']]
    print(f"page num: {num_page}; news num: {len(response['news'])}")

/var/folders/w3/fw0vkmx53s91mn8q96w593l80000gn/T/ipykernel_51195/1543417829.py:3: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text_no_html = BeautifulSoup(text, 'html.parser').get_text(separator=' ')


page num: 1; news num: 50
page num: 2; news num: 50
page num: 3; news num: 50
page num: 4; news num: 50
page num: 5; news num: 50
page num: 6; news num: 50
page num: 7; news num: 50
page num: 8; news num: 50
page num: 9; news num: 50
page num: 10; news num: 50
page num: 11; news num: 50
page num: 12; news num: 50
page num: 13; news num: 50
page num: 14; news num: 30


/var/folders/w3/fw0vkmx53s91mn8q96w593l80000gn/T/ipykernel_51195/1543417829.py:3: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text_no_html = BeautifulSoup(text, 'html.parser').get_text(separator=' ')


In [91]:
df_news['timestamp'] = pd.to_datetime(df_news['timestamp']).apply(lambda x: x.astimezone(pytz.timezone('EST')))
display(df_news.head())
df_news.shape

,text,timestamp
0,18 Stocks Moving In Monday's Pre-Market Sessio...,2016-12-05 08:29:19-05:00
1,Bank of America Upgrades ASML Holding N.V. to ...,2016-12-19 06:32:08-05:00
2,"Benzinga's Top Upgrades, Downgrades For Januar...",2017-01-17 09:22:35-05:00
3,"Earnings Scheduled For January 18, 2017\n\n \r...",2017-01-18 04:33:08-05:00
4,ASML +4.3% Premarket @$120.91; CFO Says 2017 i...,2017-01-18 07:55:31-05:00


(680, 2)

In [92]:
df_news.to_csv(f'./{symbol}_news.csv', index=False)

In [85]:
df_news.tail()

,text,datetime
675,"""ASML To Invest €100 Million A Year In Berlin ...",2023-11-20 15:22:59-05:00
676,$1000 Invested In This Stock 5 Years Ago Would...,2023-11-21 09:00:16-05:00
677,Decoding ASML Holding's Options Activity: What...,2023-11-21 15:45:48-05:00
678,Dutch Stocks Nudge Upward On Thanksgiving Afte...,2023-11-23 12:03:07-05:00
679,Evaluating ASML Holding Against Peers In Semic...,2023-11-24 11:00:28-05:00


## explore Alpha Vantage's sentiment data (in limited time range)

In [86]:
# https://www.alphavantage.co/documentation/#news-sentiment
url = 'https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers=ASML&time_from=20161125T0930&time_to=20231126T0930&apikey=xLwN8xZRExYWeGLJiGyQ&limit=1000&sort=EARLIEST'
r = requests.get(url)
data = r.json()

data

{'items': '674',
 'sentiment_score_definition': 'x <= -0.35: Bearish; -0.35 < x <= -0.15: Somewhat-Bearish; -0.15 < x < 0.15: Neutral; 0.15 <= x < 0.35: Somewhat_Bullish; x >= 0.35: Bullish',
 'relevance_score_definition': '0 < x <= 1, with a higher score indicating higher relevance.',
 'feed': [{'title': 'These 2 Nasdaq Stocks Could Carry Your Portfolio for Years',
   'url': 'https://www.fool.com/investing/2022/03/09/these-2-nasdaq-stocks-could-carry-your-portfolio-f/',
   'time_published': '20220309T111100',
   'authors': ['Harsh Chauhan'],
   'summary': 'The sell-off in these stocks gives investors the opportunity to buy two solid companies for the long run.',
   'banner_image': 'https://media.ycharts.com/charts/881eb905b73ca409f1841bc7e20fce43.png',
   'source': 'Motley Fool',
   'category_within_source': 'n/a',
   'source_domain': 'www.fool.com',
   'topics': [{'topic': 'Economy - Monetary', 'relevance_score': '0.310843'},
    {'topic': 'Retail & Wholesale', 'relevance_score': '0.

In [87]:
data.keys()

dict_keys(['items', 'sentiment_score_definition', 'relevance_score_definition', 'feed'])

In [88]:
data['feed'][-1]

{'title': 'VEA: Is Vanguard FTSE Developed Markets ETF  ( VEA )  a Solid Buy?',
 'url': 'https://stocknews.com/news/vea-dbeu-hezu-eden-asml-is-vanguard-ftse-developed-markets-etf-vea-a-solid-buy/',
 'time_published': '20230825T155221',
 'authors': ['StockNews.com Staff'],
 'summary': "The three primary U.S. stock indices closed with losses exceeding 1% each on Thursday, and investor apprehension loomed as they awaited Federal Reserve Chair Jerome Powell's speech scheduled for today.",
 'banner_image': 'https://stocknews.com/wp-content/uploads/2022/06/ETF_iSto_FB.jpg',
 'source': 'Stocknews.com',
 'category_within_source': 'n/a',
 'source_domain': 'stocknews.com',
 'topics': [{'topic': 'Economy - Monetary', 'relevance_score': '0.451494'},
  {'topic': 'Life Sciences', 'relevance_score': '0.5'},
  {'topic': 'Financial Markets', 'relevance_score': '0.999862'},
  {'topic': 'Technology', 'relevance_score': '0.5'}],
 'overall_sentiment_score': 0.125507,
 'overall_sentiment_label': 'Neutral',


In [89]:
data['feed'][-1]

{'title': 'VEA: Is Vanguard FTSE Developed Markets ETF  ( VEA )  a Solid Buy?',
 'url': 'https://stocknews.com/news/vea-dbeu-hezu-eden-asml-is-vanguard-ftse-developed-markets-etf-vea-a-solid-buy/',
 'time_published': '20230825T155221',
 'authors': ['StockNews.com Staff'],
 'summary': "The three primary U.S. stock indices closed with losses exceeding 1% each on Thursday, and investor apprehension loomed as they awaited Federal Reserve Chair Jerome Powell's speech scheduled for today.",
 'banner_image': 'https://stocknews.com/wp-content/uploads/2022/06/ETF_iSto_FB.jpg',
 'source': 'Stocknews.com',
 'category_within_source': 'n/a',
 'source_domain': 'stocknews.com',
 'topics': [{'topic': 'Economy - Monetary', 'relevance_score': '0.451494'},
  {'topic': 'Life Sciences', 'relevance_score': '0.5'},
  {'topic': 'Financial Markets', 'relevance_score': '0.999862'},
  {'topic': 'Technology', 'relevance_score': '0.5'}],
 'overall_sentiment_score': 0.125507,
 'overall_sentiment_label': 'Neutral',
